In [12]:
import pygame
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.181")

pygame.init()
WIDTH, HEIGHT = 600, 400
screen = pygame.display.set_mode((WIDTH, HEIGHT))

ROAD_WIDTH = 400 
ROAD_LEFT = (WIDTH - ROAD_WIDTH) // 2
ROAD_RIGHT = ROAD_LEFT + ROAD_WIDTH
LANE_MARK_WIDTH = 8
LANE_MARK_HEIGHT = 40
LANE_MARK_SPACING = 30

MAX_X = ROAD_RIGHT - 20
MIN_X = ROAD_LEFT + 20
MAX_Y = HEIGHT - 20
MIN_Y = 20

def draw_road(scroll_offset):
    screen.fill((80, 170, 80)) # background
    pygame.draw.rect(screen, (40, 40, 40), (ROAD_LEFT, 0, ROAD_WIDTH, HEIGHT)) # road
    pygame.draw.rect(screen, (255, 255, 255), (ROAD_LEFT, 0, 10, HEIGHT)) # left line
    pygame.draw.rect(screen, (255, 255, 255), (ROAD_RIGHT-10, 0, 10, HEIGHT)) # right line

    for y in range(-LANE_MARK_HEIGHT, HEIGHT, LANE_MARK_HEIGHT+LANE_MARK_SPACING):
        draw_y = y + scroll_offset % (LANE_MARK_HEIGHT + LANE_MARK_SPACING)
        pygame.draw.rect(screen, (255, 255, 0), (WIDTH//2-LANE_MARK_WIDTH//2, draw_y,
                        LANE_MARK_WIDTH, LANE_MARK_HEIGHT))

def draw_obstacle(obstacle):
    pygame.draw.rect(screen, (30, 30, 180), obstacle)
    pygame.draw.rect(screen, (255, 255, 255), 2) # outline

x, y = WIDTH // 2, HEIGHT // 2 
speed = 7

def clamp(value, low, high):
    """Clamp a value between the low and high."""
    return max(low, min(high, value))

def read_gyro():
    data = got.read_gyro_data()
    pitch = data[0]
    roll = data[1]
    yaw = data[2]
    return pitch, roll, yaw

center_pitch, center_roll, center_yaw = read_gyro()
scroll_offset = 0
obstacles = []

running = True
while running:
    scroll_offset += speed
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        
        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                center_pitch, center_roll, center_yaw = read_gyro()
                x, y = WIDTH // 2, HEIGHT // 2 

    pitch, roll, yaw = read_gyro()

    move_x = (roll - center_roll) / 20 # left / right
    move_y = (pitch - center_pitch) / -20 # up / down

    x += move_x * speed
    y += move_y * speed

    x = clamp(x, MIN_X, MAX_X)
    y = clamp(y, MIN_Y, MAX_Y)

    # screen.fill((255, 255, 255))
    draw_road(scroll_offset)
    pygame.draw.circle(screen, (0, 255, 0), (int(x), int(y)), 20)

    pygame.display.flip()

pygame.quit()

192.168.1.181:50051


In [ ]:
from ugot import ugot
got = ugot.UGOT()
got.initialize("192.168.1.181")
got.load_models(["apriltag_qrcode"])

got.balance_start_balancing()
tags = got.get_apriltag_total_info()
input() # wait for user input before moving
while not tags:
    got.balance_move_speed(0, 10)
    tags = got.get_apriltag_total_info()
count = 0 # buffer to ensure apriltag is really gone
run = True
while run:
    got.balance_move_speed(0, 80)
    tags = got.get_apriltag_total_info()
    if not tags:
        count += 1
        if count > 2: # loop has run 3 times without seeing apriltag
            run = False
    else:
        count = 0 # reset count if apriltag is seen again
got.balance_stop_balancing()
# 1 metre to stop 

192.168.1.181:50051
